# 🌾 Crop Damage Insurance Assessment System (v2 — FAST)

## Research-Backed AI Insurance Pipeline — ⚡ Optimized for Speed
- **EfficientNet B3** backbone (upgraded from B0)
- **GradCAM** heatmaps for explainability
- **Full dataset training** (all 26,068 images — no sampling)
- **Detailed accuracy report** with confusion matrix
- **Test set predictions** for submission

### ⚡ Speed Optimizations Applied:
- ✅ **Local SSD copy** — images copied from Drive to local `/content/` (10–50× I/O speedup)
- ✅ **Mixed Precision (AMP)** — FP16 training (~2× GPU throughput on T4)
- ✅ **BATCH_SIZE=64** — fully utilizes T4’s 15GB VRAM
- ✅ **NUM_WORKERS=4** — parallel data loading (was 0!)
- ✅ **cuDNN benchmark** — auto-tuned convolution kernels
- ✅ **Faster optimizer step** — `set_to_none=True`

**Expected: ~5–15 min/epoch (was 4 hours!)**

---

### Outputs:
✅ `classifier_model.pth` — trained model weights  
✅ `config.pkl` / `config.json` — model configuration  
✅ GradCAM heatmap visualizations  
✅ Detailed classification report + confusion matrix  
✅ Test set predictions (`submission.csv`)

In [ ]:
# Install dependencies
!pip install -q timm tqdm scikit-learn scikit-image seaborn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted!")

In [ ]:
# Core imports
import os, gc, json, random, pickle, shutil, time
from datetime import datetime, date
from collections import Counter
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

# ⚡ SPEED: Enable cuDNN auto-tuner for fastest convolution kernels
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
    print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")

## ⚡ Copy Images to Local SSD (CRITICAL SPEED FIX)

In [ ]:
# ⚡ SPEED FIX #1: Copy images from slow Drive to fast local SSD
# Google Drive is a network FUSE mount — random reads are 10-50x slower than local disk.
# This one-time copy takes ~2-5 minutes but saves HOURS during training.

DRIVE_IMG_DIR = "/content/drive/MyDrive/images"
LOCAL_IMG_DIR = "/content/local_images"

if not os.path.exists(LOCAL_IMG_DIR):
    print("⏳ Copying images to local SSD for fast I/O...")
    start = time.time()
    shutil.copytree(DRIVE_IMG_DIR, LOCAL_IMG_DIR)
    elapsed = time.time() - start
    print(f"✅ Copied {len(os.listdir(LOCAL_IMG_DIR))} files to local SSD in {elapsed:.0f}s")
else:
    print(f"✅ Local images already exist ({len(os.listdir(LOCAL_IMG_DIR))} files)")

## ⚙️ Configuration

In [ ]:
class Config:
    # ===========================================
    # 🔴 YOUR DRIVE PATHS (verified working)
    # ===========================================
    DRIVE_PATH = "/content/drive/MyDrive"
    CSV_DIR = f"{DRIVE_PATH}/ColabNotebooks"

    # ⚡ SPEED: Use LOCAL copy of images instead of Drive
    IMG_DIR = "/content/local_images"   # ← fast local SSD

    TRAIN_CSV = f"{CSV_DIR}/Train.csv"
    TEST_CSV = f"{CSV_DIR}/Test.csv"

    OUTPUT_DIR = "/content/outputs"
    MODELS_DIR = f"{OUTPUT_DIR}/models"
    RESULTS_DIR = f"{OUTPUT_DIR}/results"
    HEATMAP_DIR = f"{OUTPUT_DIR}/heatmaps"

    # ===========================================
    # ⚡ Training settings (OPTIMIZED for speed)
    # ===========================================
    SEED = 42
    N_FOLDS = 3
    SAMPLE_SIZE = 0            # ← 0 = USE ALL DATA (no limit!)
    BATCH_SIZE = 64            # ⚡ was 16 — T4 handles 64 easily with AMP
    IMG_SIZE = 224
    EPOCHS = 12
    LEARNING_RATE = 2e-4
    NUM_WORKERS = 4            # ⚡ was 0 — parallel data loading
    PATIENCE = 3

    # v2: upgraded backbone
    BACKBONE = 'efficientnet_b3'

    # Damage classes
    DAMAGE_CLASSES = ['DR', 'G', 'ND', 'WD', 'other']
    DAMAGE_NAMES = {
        'DR': 'Drought', 'G': 'Good/Healthy',
        'ND': 'Nutrient Deficiency', 'WD': 'Weed Damage',
        'other': 'Other Damage'
    }
    NUM_CLASSES = 5

    # CGIAR → PBI mapping
    CGIAR_TO_PBI = {
        'DR': 'drought', 'G': 'healthy',
        'ND': 'disease', 'WD': 'pest', 'other': 'other'
    }

    DEFAULT_IMAGE_COVERAGE_M2 = 1500.0
    AREA_VARIANCE_FACTOR = 0.15
    OVERLAP_SSIM_THRESHOLD = 0.6
    GRADCAM_THRESHOLD = 0.5
    CNN_WEIGHT = 0.7
    RGB_WEIGHT = 0.3

# Create directories
for d in [Config.OUTPUT_DIR, Config.MODELS_DIR, Config.RESULTS_DIR, Config.HEATMAP_DIR]:
    os.makedirs(d, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def set_seed(seed=Config.SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed()

# Verify data
train_df = pd.read_csv(Config.TRAIN_CSV)
print(f"✓ Config loaded!")
print(f"  Backbone      : {Config.BACKBONE}")
print(f"  Sample limit  : {Config.SAMPLE_SIZE} (0 = ALL DATA)")
print(f"  Batch size    : {Config.BATCH_SIZE} (⚡ optimized)")
print(f"  Num workers   : {Config.NUM_WORKERS} (⚡ parallel loading)")
print(f"  Mixed Precision: ✅ Enabled (AMP FP16)")
print(f"  Image source  : {Config.IMG_DIR} (⚡ local SSD)")
print(f"  Training data : {len(train_df)} images")
print(f"  Class distribution:")
for cls, count in train_df['damage'].value_counts().items():
    print(f"    {cls:>6s}: {count:>6d} ({100*count/len(train_df):.1f}%)")
del train_df

## 🧠 CropDamageClassifier (v2 — EfficientNet B3)

In [ ]:
class CropDamageClassifier(nn.Module):
    """EfficientNet-based crop damage classifier (v2)"""
    def __init__(self, backbone=Config.BACKBONE, num_classes=Config.NUM_CLASSES, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool='avg')
        with torch.no_grad():
            feat_dim = self.backbone(torch.randn(1, 3, Config.IMG_SIZE, Config.IMG_SIZE)).shape[1]
        self.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(feat_dim, 256), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.backbone(x))

print("✓ CropDamageClassifier defined — backbone:", Config.BACKBONE)

## 🔥 GradCAM — Explainability

In [ ]:
class GradCAM:
    """Grad-CAM heatmap generator."""
    def __init__(self, model):
        self.model = model; self.model.eval()
        self._act = self._grad = None
        layer = self._find_last_conv(model.backbone)
        layer.register_forward_hook(lambda m,i,o: setattr(self, '_act', o.detach()))
        layer.register_backward_hook(lambda m,gi,go: setattr(self, '_grad', go[0].detach()))
    @staticmethod
    def _find_last_conv(module):
        last = None
        for m in module.modules():
            if isinstance(m, nn.Conv2d): last = m
        if last is None: raise RuntimeError("No Conv2d")
        return last
    def generate(self, x, target=None):
        self.model.eval(); x = x.clone().requires_grad_(True)
        out = self.model(x)
        if target is None: target = out.argmax(1).item()
        self.model.zero_grad(); out[0, target].backward()
        w = self._grad.mean(dim=[2,3], keepdim=True)
        cam = F.relu((w * self._act).sum(1, keepdim=True)).squeeze().cpu().numpy()
        if cam.max() > 0: cam /= cam.max()
        return cv2.resize(cam, (Config.IMG_SIZE, Config.IMG_SIZE))
    @staticmethod
    def overlay(img, hm, alpha=0.5):
        c = cv2.applyColorMap((hm*255).astype(np.uint8), cv2.COLORMAP_JET)
        return (img*(1-alpha) + cv2.cvtColor(c, cv2.COLOR_BGR2RGB)*alpha).astype(np.uint8)
    @staticmethod
    def damage_ratio(hm, t=0.5): return float((hm>=t).sum())/hm.size

print("✓ GradCAM ready!")

## 📂 Dataset & Transforms

In [ ]:
class CropDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir; self.transform = transform; self.is_test = is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.img_dir, row['filename'])
        try:
            # ⚡ SPEED: Use cv2 for faster image decoding, then convert to PIL for transforms
            img_bgr = cv2.imread(path)
            if img_bgr is not None:
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(img_rgb)
            else:
                img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (Config.IMG_SIZE, Config.IMG_SIZE), (128,128,128))
        if self.transform: img = self.transform(img)
        if self.is_test: return img, row['ID']
        return img, torch.tensor(row['label'], dtype=torch.long)

def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

print("✓ Dataset ready!")

## 🚀 Training Pipeline (Full Dataset — ⚡ FAST with AMP)

In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def train_classifier():
    """Train on ALL data — no sampling limit. ⚡ Optimized with AMP + fast I/O."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"{'='*60}")
    print(f"🌾 TRAINING — {Config.BACKBONE} (⚡ FAST MODE)")
    print(f"{'='*60}")
    print(f"Device: {device} | Epochs: {Config.EPOCHS} | BS: {Config.BATCH_SIZE}")
    print(f"Workers: {Config.NUM_WORKERS} | AMP: ✅ | cuDNN benchmark: ✅")
    print(f"Image source: {Config.IMG_DIR}")

    df = pd.read_csv(Config.TRAIN_CSV)
    print(f"\n📊 Full dataset: {len(df)} images (NO SAMPLING)")
    print(df["damage"].value_counts())

    # NO SAMPLING — use all data
    label_map = {c: i for i, c in enumerate(Config.DAMAGE_CLASSES)}
    df['label'] = df['damage'].map(label_map).fillna(4).astype(int)

    skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
    df['fold'] = -1
    for fold, (_, vi) in enumerate(skf.split(df, df['label'])):
        df.loc[vi, 'fold'] = fold

    # Track all predictions across folds (OOF = out-of-fold)
    oof_preds = np.zeros(len(df), dtype=int)
    oof_probs = np.zeros((len(df), Config.NUM_CLASSES))
    fold_accs = []
    fold_reports = []

    for fold in range(Config.N_FOLDS):
        print(f"\n{'='*50}")
        print(f"FOLD {fold+1}/{Config.N_FOLDS}")
        print(f"{'='*50}")

        t_df = df[df['fold'] != fold]
        v_df = df[df['fold'] == fold]
        v_idx = v_df.index.values
        print(f"Train: {len(t_df)} | Val: {len(v_df)}")

        # ⚡ SPEED: persistent_workers keeps workers alive between epochs
        tl = DataLoader(CropDataset(t_df, Config.IMG_DIR, get_transforms(True)),
            batch_size=Config.BATCH_SIZE, shuffle=True,
            num_workers=Config.NUM_WORKERS, pin_memory=True,
            persistent_workers=True if Config.NUM_WORKERS > 0 else False,
            prefetch_factor=2 if Config.NUM_WORKERS > 0 else None)
        vl = DataLoader(CropDataset(v_df, Config.IMG_DIR, get_transforms(False)),
            batch_size=Config.BATCH_SIZE, shuffle=False,
            num_workers=Config.NUM_WORKERS, pin_memory=True,
            persistent_workers=True if Config.NUM_WORKERS > 0 else False,
            prefetch_factor=2 if Config.NUM_WORKERS > 0 else None)

        model = CropDamageClassifier().to(device)
        criterion = nn.CrossEntropyLoss()
        opt = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, Config.EPOCHS)

        # ⚡ SPEED: Mixed Precision scaler
        scaler = torch.amp.GradScaler('cuda')

        best_acc, patience = 0, 0
        start_epoch = 0
        chkpt_path = f"{Config.MODELS_DIR}/checkpoint_fold{fold}.pth"
        drive_chkpt_path = f"{Config.DRIVE_PATH}/checkpoint_fold{fold}.pth"

        if os.path.exists(drive_chkpt_path) and not os.path.exists(chkpt_path):
            shutil.copy(drive_chkpt_path, chkpt_path)

        if os.path.exists(chkpt_path):
            checkpoint = torch.load(chkpt_path, map_location=device)
            model.load_state_dict(checkpoint['model_state'])
            opt.load_state_dict(checkpoint['optimizer_state'])
            start_epoch = checkpoint['epoch'] + 1
            best_acc = checkpoint.get('best_acc', 0)
            if 'scaler_state' in checkpoint:
                scaler.load_state_dict(checkpoint['scaler_state'])
            print(f"Resuming training from epoch {start_epoch}")

        for epoch in range(start_epoch, Config.EPOCHS):
            epoch_start = time.time()
            model.train()
            tc, tt, tl_sum = 0, 0, 0.0

            for imgs, labels in tqdm(tl, desc=f"E{epoch+1}"):
                imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

                # ⚡ SPEED: Mixed Precision forward pass (FP16 on GPU)
                opt.zero_grad(set_to_none=True)  # faster than zero_grad()
                with torch.amp.autocast('cuda'):
                    out = model(imgs)
                    loss = criterion(out, labels)

                # ⚡ SPEED: Scaled backward + optimizer step
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

                tc += (out.argmax(1)==labels).sum().item()
                tt += labels.size(0)
                tl_sum += loss.item()*labels.size(0)

            model.eval()
            vp, vl_list, vc, vt = [], [], 0, 0
            vprobs = []
            with torch.no_grad():
                # ⚡ SPEED: AMP autocast for validation too (faster inference)
                with torch.amp.autocast('cuda'):
                    for imgs, labels in vl:
                        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                        out = model(imgs)
                        probs = F.softmax(out, dim=1).cpu().numpy()
                        preds = out.argmax(1)
                        vc += (preds==labels).sum().item()
                        vt += labels.size(0)
                        vp.extend(preds.cpu().numpy()); vl_list.extend(labels.cpu().numpy())
                        vprobs.append(probs)

            sched.step()
            va = 100*vc/vt
            epoch_time = time.time() - epoch_start
            print(f"  Train: {100*tc/tt:.1f}% Loss: {tl_sum/tt:.4f} | Val: {va:.1f}% | ⏱ {epoch_time:.0f}s ({epoch_time/60:.1f}min)")

            if va > best_acc:
                best_acc = va; patience = 0
                torch.save(model.state_dict(), f"{Config.MODELS_DIR}/classifier_fold{fold}.pth")
                best_preds = np.array(vp)
                best_probs = np.concatenate(vprobs)
                best_labels = np.array(vl_list)
                print(f"  ✓ Saved (val_acc={va:.1f}%)")
            else:
                patience += 1
                if patience >= Config.PATIENCE: print("  Early stopping"); break

            # Save checkpoint (includes scaler state for AMP)
            chkpt_data = {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_acc": best_acc,
            }
            torch.save(chkpt_data, chkpt_path)
            torch.save(chkpt_data, drive_chkpt_path)

        # Store OOF predictions
        oof_preds[v_idx] = best_preds
        oof_probs[v_idx] = best_probs
        fold_accs.append(best_acc)

        # Per-fold report
        rpt = classification_report(best_labels, best_preds, target_names=Config.DAMAGE_CLASSES, zero_division=0)
        print(f"\nFold {fold+1} Report:\n{rpt}")
        fold_reports.append(rpt)

        del model, opt, scaler; clear_memory()

    # Save best fold
    bf = int(np.argmax(fold_accs))
    shutil.copy(f"{Config.MODELS_DIR}/classifier_fold{bf}.pth", f"{Config.MODELS_DIR}/classifier_model.pth")

    # Save config
    cfg = {
        'DAMAGE_CLASSES': Config.DAMAGE_CLASSES, 'DAMAGE_NAMES': Config.DAMAGE_NAMES,
        'CGIAR_TO_PBI': Config.CGIAR_TO_PBI, 'IMG_SIZE': Config.IMG_SIZE,
        'BACKBONE': Config.BACKBONE, 'NUM_CLASSES': Config.NUM_CLASSES,
        'DEFAULT_IMAGE_COVERAGE_M2': Config.DEFAULT_IMAGE_COVERAGE_M2,
        'CNN_WEIGHT': Config.CNN_WEIGHT, 'RGB_WEIGHT': Config.RGB_WEIGHT,
        'GRADCAM_THRESHOLD': Config.GRADCAM_THRESHOLD,
        'trained_at': datetime.now().isoformat(),
        'best_fold': bf, 'best_accuracy': fold_accs[bf],
        'fold_accuracies': fold_accs, 'total_train_images': len(df),
    }
    with open(f"{Config.MODELS_DIR}/config.pkl", "wb") as f: pickle.dump(cfg, f)
    with open(f"{Config.MODELS_DIR}/config.json", "w") as f: json.dump(cfg, f, indent=2)

    return df, oof_preds, oof_probs, fold_accs, fold_reports


In [ ]:
# 🚀 TRAIN ON ALL DATA (⚡ FAST)
training_start = time.time()
try:
    df, oof_preds, oof_probs, fold_accs, fold_reports = train_classifier()
    total_time = time.time() - training_start
    print(f"\n🎉 Total training time: {total_time/60:.1f} min ({total_time/3600:.1f} hr)")
except Exception as e:
    print("Training error:", e)
    import traceback; traceback.print_exc()

## 📊 Detailed Accuracy Report (Elaborative)

In [ ]:
# ============================================================
# 📊 ELABORATIVE ACCURACY REPORT
# ============================================================
true_labels = df['label'].values

print('=' * 70)
print('🌾 OVERALL MODEL PERFORMANCE (Out-of-Fold on ALL 26,068 images)')
print('=' * 70)

# Overall metrics
oof_acc = accuracy_score(true_labels, oof_preds)
oof_f1_macro = f1_score(true_labels, oof_preds, average='macro', zero_division=0)
oof_f1_weighted = f1_score(true_labels, oof_preds, average='weighted', zero_division=0)
oof_prec = precision_score(true_labels, oof_preds, average='weighted', zero_division=0)
oof_rec = recall_score(true_labels, oof_preds, average='weighted', zero_division=0)

print(f"\n📈 Overall Accuracy     : {oof_acc:.2%}")
print(f"📈 Weighted F1-Score    : {oof_f1_weighted:.4f}")
print(f"📈 Macro F1-Score       : {oof_f1_macro:.4f}")
print(f"📈 Weighted Precision   : {oof_prec:.4f}")
print(f"📈 Weighted Recall      : {oof_rec:.4f}")

# Per-fold summary
print(f"\n{'─'*50}")
print('Per-Fold Accuracy:')
print(f"{'─'*50}")
for i, acc in enumerate(fold_accs):
    bar = "█" * int(acc / 2) + "░" * (50 - int(acc / 2))
    print(f"  Fold {i+1}: {acc:6.2f}%  {bar}")
print(f"  Average: {np.mean(fold_accs):.2f}%")
print(f"  Std Dev: {np.std(fold_accs):.2f}%")

# Full classification report
print(f"\n{'─'*50}")
print('Full Classification Report (OOF):')
print(f"{'─'*50}")
print(classification_report(true_labels, oof_preds, target_names=Config.DAMAGE_CLASSES, zero_division=0))

# Per-class breakdown
print(f"\n{'─'*50}")
print('Per-Class Breakdown:')
print(f"{'─'*50}")
for i, cls in enumerate(Config.DAMAGE_CLASSES):
    mask = true_labels == i
    if mask.sum() == 0: continue
    cls_acc = (oof_preds[mask] == i).sum() / mask.sum()
    cls_count = int(mask.sum())
    print(f"  {Config.DAMAGE_NAMES[cls]:>22s} ({cls}): {cls_acc:.1%} accuracy on {cls_count} samples")


## 📉 Confusion Matrix

In [ ]:
# Confusion Matrix visualization
cm = confusion_matrix(true_labels, oof_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=Config.DAMAGE_CLASSES, yticklabels=Config.DAMAGE_CLASSES, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14)
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted')

# Normalized (percentages)
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='YlOrRd',
    xticklabels=Config.DAMAGE_CLASSES, yticklabels=Config.DAMAGE_CLASSES, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized %)', fontsize=14)
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f"{Config.RESULTS_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Confusion matrix saved!")

## 🔥 GradCAM Visualization

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CropDamageClassifier(pretrained=False).to(device)
model.load_state_dict(torch.load(f"{Config.MODELS_DIR}/classifier_model.pth", map_location=device))
model.eval()
cam = GradCAM(model)

samples = pd.read_csv(Config.TRAIN_CSV)
samples = samples.groupby('damage').apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)

fig, axes = plt.subplots(len(samples), 3, figsize=(15, 4*len(samples)))
tf = get_transforms(False)

for i, (_, row) in enumerate(samples.iterrows()):
    img_path = os.path.join(Config.IMG_DIR, row['filename'])
    orig = np.array(Image.open(img_path).convert('RGB').resize((Config.IMG_SIZE, Config.IMG_SIZE)))
    tensor = tf(Image.open(img_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1).cpu().numpy()[0]
    pi = int(np.argmax(probs))
    hm = cam.generate(tensor, pi)
    ov = cam.overlay(orig, hm)
    ratio = cam.damage_ratio(hm)
    axes[i,0].imshow(orig); axes[i,0].set_title(f"True: {row['damage']}")
    axes[i,1].imshow(hm, cmap="jet"); axes[i,1].set_title("GradCAM")
    axes[i,2].imshow(ov); axes[i,2].set_title(f"Pred: {Config.DAMAGE_CLASSES[pi]} ({probs[pi]:.0%}) Area:{ratio:.0%}")
    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.savefig(f"{Config.RESULTS_DIR}/gradcam_samples.png", dpi=150)
plt.show()
print("✓ GradCAM visualization saved!")

## 📝 Test Set Predictions (submission.csv)

In [ ]:
# Generate predictions for the test set
test_df = pd.read_csv(Config.TEST_CSV)
print(f"Test set: {len(test_df)} images")

test_ds = CropDataset(test_df, Config.IMG_DIR, get_transforms(False), is_test=True)
test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

model.eval()
test_preds = []
test_ids = []

with torch.no_grad():
    with torch.amp.autocast('cuda'):
        for imgs, ids in tqdm(test_loader, desc='Predicting'):
            imgs = imgs.to(device)
            out = model(imgs)
            preds = out.argmax(1).cpu().numpy()
            test_preds.extend(preds)
            test_ids.extend(ids)

# Create submission
sub = pd.DataFrame({'ID': test_ids, 'damage': [Config.DAMAGE_CLASSES[p] for p in test_preds]})
sub.to_csv(f"{Config.RESULTS_DIR}/submission.csv", index=False)

# Also save to Drive
sub.to_csv(f"{Config.DRIVE_PATH}/ColabNotebooks/submission.csv", index=False)

print(f"\n✓ Submission saved! ({len(sub)} predictions)")
print(f"  Test prediction distribution:")
for cls, count in sub['damage'].value_counts().items():
    print(f"    {cls:>6s}: {count:>5d} ({100*count/len(sub):.1f}%)")

## 💾 Export Model to Drive

In [ ]:
drive_out = f"{Config.DRIVE_PATH}/crop_damage_models"
os.makedirs(drive_out, exist_ok=True)

for fname in ['classifier_model.pth', 'config.pkl', 'config.json']:
    src = f"{Config.MODELS_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, os.path.join(drive_out, fname))
        print(f"✓ {fname} → {drive_out}")

# Also copy confusion matrix and GradCAM
for fname in ["confusion_matrix.png", "gradcam_samples.png"]:
    src = f"{Config.RESULTS_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy(src, os.path.join(drive_out, fname))

print(f"\n🎉 All files saved to: {drive_out}")
print("\n📋 TO DEPLOY IN YOUR BACKEND:")
print("  1. Download classifier_model.pth + config.pkl from Drive")
print("  2. Place BOTH files in: cropfarmPY/models/")
print("  3. The pipeline auto-detects and uses them!")
print("  (config.pkl tells the pipeline which backbone + classes to use)")